# BMI Prediction from Silhouette
This notebook trains and evaluates a hybrid CNN + Geometric Feature PyTorch model to predict BMI from binary silhouette masks.

In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

DATA_DIR = Path("train")
MASK_DIR = DATA_DIR / "mask_clean"
MODEL_PATH = Path("bmi_model.pth")
CHECKPOINT_PATH = Path("bmi_model_checkpoint.pth")
CACHE_PATH = DATA_DIR / "cached_features_and_images.pt"

device = torch.device("cpu")
print("Using device:", device)

### 1. Geometric Feature Extraction
Extracts 14 scale-invariant features without destroying thickness signals:
- : area / height^2
- : width / height
- : area / (width * height)
- 11 slice widths at 0% to 100% body height, normalized by height ().

In [2]:
def extract_geometric_features(mask):
    y, x = np.where(mask > 127)
    if len(x) == 0:
        return None
    area = len(x)
    y_min, y_max = y.min(), y.max()
    height = y_max - y_min + 1
    x_min, x_max = x.min(), x.max()
    width = x_max - x_min + 1

    normalized_ratio = float(area / (height ** 2))
    aspect_ratio = float(width / height)
    density = float(area / (width * height))

    def get_largest_segment_width(row, gap_threshold=5):
        x_vals = np.where(row > 127)[0]
        if len(x_vals) == 0:
            return 0
        gaps = np.diff(x_vals)
        break_indices = np.where(gaps > gap_threshold)[0]
        segments = np.split(x_vals, break_indices + 1)
        largest = max(segments, key=len)
        return float(largest[-1] - largest[0] + 1)

    width_values = []
    for i in range(0, 101, 10):
        search_level = int(np.clip(y_min + (y_max - y_min) * (i / 100.0), 0, mask.shape[0] - 1))
        row = mask[search_level]
        w = get_largest_segment_width(row)
        width_values.append(float(w / height))

    return [normalized_ratio, aspect_ratio, density] + width_values

### 2. Model Architecture
Modern CNN backbone with , Global Average Pooling (), and .

In [3]:
class BMIModel(nn.Module):
    def __init__(self, num_geometry_features=14):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.cnn_fc = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.geometry_fc = nn.Sequential(
            nn.Linear(num_geometry_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )
        self.final_fc = nn.Sequential(
            nn.Linear(32 + 16, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, image, geometry):
        x = self.cnn(image)
        x = x.flatten(1)
        x = self.cnn_fc(x)
        g = self.geometry_fc(geometry)
        combined = torch.cat((x, g), dim=1)
        return self.final_fc(combined)

### 3. Load Trained Model & Evaluation Metrics
Loads the trained production checkpoint and evaluates performance on the held-out validation set.

In [4]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model = BMIModel(num_geometry_features=checkpoint["num_geometry_features"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully!")
metrics = checkpoint.get("metrics", {})
for k, v in metrics.items():
    print(f"{k.upper()}: {v:.3f}")

### 4. Single-Image Inference Function
Predicts BMI for any binary silhouette (from webcam or file).

In [5]:
from predict_bmi import BMIPredictor

predictor = BMIPredictor(CHECKPOINT_PATH)

sample_img = MASK_DIR / "0b8fa4d88d8a238a483d382b8f787372.png"
pred = predictor.predict(sample_img)
print(f"Image: {sample_img.name}")
print(f"Predicted BMI: {pred:.2f}")